# FireChronicle — MODIS and VIIRS annual processing

**Region:** Pará, Brazil (`BR-PA`)  
**Processing period:** 2012–2020  
**Spatial reference:** EPSG:6933  
**Grid:** 50 km equal-area blocks

This notebook processes one calendar year at a time from the original NASA FIRMS Brazil country CSV files. Aqua MODIS and Suomi-NPP VIIRS detections are filtered independently, clipped to the Pará boundary, and assigned to the project's canonical 50 km grid.

The notebook creates the annual datasets used by the FireChronicle backend and data-science workflow. It also writes an audit file containing source-file hashes, output-file hashes, row counts, and validation results.

## What this notebook does

1. Reads the annual Aqua MODIS and S-NPP VIIRS source CSVs.
2. Checks the IBGE Pará boundary and builds the 50 km equal-area grid.
3. Filters each sensor separately and aggregates detections by block, month, and day/night.
4. Builds the master, sparse ML, backend monthly, coverage, and full-panel datasets.
5. Validates the output schemas, keys, detection totals, and panel dimensions.
6. Exports the annual package and its audit report.

## Data handling

- MODIS and VIIRS counts remain separate; they are never added together.
- A missing detection record is not automatically interpreted as zero.
- Zero conversion is allowed only when the optional completeness file explicitly supports it.
- The output coverage status remains `unknown` where observation coverage has not been independently verified.
- The source CSVs are read only; the pipeline does not modify them.

## Required input files

For the selected year, place these files in `/content`:

- `modis_YEAR_Brazil.csv`
- `viirs-snpp_YEAR_Brazil.csv`
- `PA_UF_2024.zip`

An optional `source_period_completeness_YEAR.csv` file can provide period-level completeness evidence. Without it, the pipeline uses conservative defaults and leaves unverified counts as NULL.

**Source archive:** [NASA FIRMS country data](https://firms.modaps.eosdis.nasa.gov/country/)

> The original source data and generated annual packages are not embedded in this notebook. Keep large data files outside the Git repository and upload them when running the notebook in Colab.

[Open in Google Colab](https://colab.research.google.com/github/inan001/firechronicle/blob/main/notebooks/firechronicle_modis_viirs_2012_2020.ipynb)

## 1. Install the required packages

Run this cell once in a fresh Colab runtime. Shapely 2 is required for the vectorized point-in-polygon operation used by the pipeline.

In [ ]:
%pip -q install pandas numpy pyarrow "geopandas>=0.14" "shapely>=2" pyproj

## 2. Select the year and set the input/output paths

Change `YEAR` to the year you want to process. The output folder is named for that year.

The pipeline deliberately refuses to overwrite an existing output folder. If you need to run the same year again, review the existing results first and choose a new `OUTPUT_ROOT`.

In [ ]:
from pathlib import Path
import os

YEAR = 2012  # Select one year from 2012 through 2020.
assert YEAR in range(2012, 2021), "YEAR must be between 2012 and 2020."

INPUT_DIR = "/content"
OUTPUT_ROOT = f"/content/firechronicle_{YEAR}_verified_v1"
EXPORT_FORMAT = "parquet"  # Use "csv" only when a CSV preview is needed.

os.environ["FC_YEAR"] = str(YEAR)
os.environ["FC_INPUT_DIR"] = INPUT_DIR
os.environ["FC_OUTPUT_DIR"] = OUTPUT_ROOT
os.environ["FC_EXPORT_FORMAT"] = EXPORT_FORMAT

print(f"Year: {YEAR}")
print(f"MODIS source: modis_{YEAR}_Brazil.csv")
print(f"VIIRS source: viirs-snpp_{YEAR}_Brazil.csv")
print("Boundary: PA_UF_2024.zip")
print(f"Output folder: {OUTPUT_ROOT}")
print(f"Export format: {EXPORT_FORMAT}")

## 3. Upload the source files

Upload the two annual NASA FIRMS CSV files and the approved IBGE boundary ZIP. Keep the filenames exactly as shown in the previous cell.

If you have a completeness file, upload it as `source_period_completeness_YEAR.csv` as well. It is optional; the default pipeline does not infer zero counts without documented evidence.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

required_files = [
    f"modis_{YEAR}_Brazil.csv",
    f"viirs-snpp_{YEAR}_Brazil.csv",
    "PA_UF_2024.zip",
]
missing_files = [
    name for name in required_files
    if not Path(INPUT_DIR, name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required files were not found in INPUT_DIR: "
        f"{missing_files}. Check the filenames and Colab Files panel."
    )

optional_completeness = Path(
    INPUT_DIR, f"source_period_completeness_{YEAR}.csv"
)
if optional_completeness.is_file():
    print("Completeness evidence found:", optional_completeness.name)
else:
    print("No optional completeness file found; conservative NULL handling will be used.")

print("All required source files are available.")

## 4. Annual processing code

The following code is the processing implementation used by this notebook. It is shown as ordinary Python so the functions and validation steps can be reviewed directly on GitHub.

The processing logic is retained from the supplied notebook. The changes here are limited to the notebook presentation and the stale 2019 label/defaults in the embedded script.

In [ ]:
"""FireChronicle annual MODIS–VIIRS source-to-exports pipeline. Original Brazil FIRMS CSV inputs are never changed."""
from pathlib import Path
import os, json, hashlib, zipfile, shutil, tempfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from shapely import intersects_xy
from pyproj import Transformer

BASE=Path(os.environ.get('FC_INPUT_DIR','/mnt/data'))
OUT=Path(os.environ.get('FC_OUTPUT_DIR',f'/content/firechronicle_{int(os.environ.get("FC_YEAR","2012"))}_verified_v1'))
YEAR=int(os.environ.get('FC_YEAR','2012'))
MODIS=BASE/f'modis_{YEAR}_Brazil.csv'; VIIRS=BASE/f'viirs-snpp_{YEAR}_Brazil.csv'; BOUNDARY=BASE/'PA_UF_2024.zip'
COMPLETENESS=BASE/f'source_period_completeness_{YEAR}.csv'
MASTER_COLS=['region_id','block_id','block_ix','block_iy','year','month','daynight','modis_count','viirs_count','modis_frp_sum','viirs_frp_sum','coverage_status','processing_status','modis_source_version','viirs_source_version']
ML_COLS=['region_id','block_id','block_ix','block_iy','year','month','daynight','modis_count','viirs_count','coverage_status']
BACK_COLS=['region_id','block_id','block_ix','block_iy','year','month','modis_aqua_count','viirs_snpp_count','ratio_viirs_to_modis','log_ratio','modis_frp_sum','viirs_frp_sum']
COV_COLS=['region_id','block_id','year','month','daynight','sensor','processing_status','coverage_status','source_product','source_version','observation_evidence','notes']
KEY=['region_id','block_id','block_ix','block_iy','year','month','daynight']
SKEY=['region_id','block_id','year','month','daynight']

def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
 return h.hexdigest()

def require(cond,msg):
 if not bool(cond):raise AssertionError(msg)

def load_boundary():
 with zipfile.ZipFile(BOUNDARY) as z:
  names={Path(x).name for x in z.namelist()}
  require(set('PA_UF_2024.'+e for e in ['shp','shx','dbf','prj']).issubset(names),'Missing shapefile component')
  import tempfile
  with tempfile.TemporaryDirectory() as td:
   for member in z.namelist():
    if Path(member).name.startswith('PA_UF_2024.') and Path(member).suffix.lower() in {'.shp','.shx','.dbf','.prj','.cpg'}:
     (Path(td)/Path(member).name).write_bytes(z.read(member))
   raw=gpd.read_file(str(Path(td)/'PA_UF_2024.shp'))
 require(len(raw)==1 and raw['CD_UF'].astype(str).iloc[0]=='15','Boundary must be exactly Pará CD_UF=15')
 require(raw.crs is not None and raw.crs.to_epsg()==4674,'Unexpected boundary CRS')
 require(raw.geometry.is_valid.all(),'Invalid boundary geometry')
 pa4326=raw.to_crs(4326).geometry.iloc[0]
 pa6933=raw.to_crs(6933).geometry.iloc[0]
 return pa4326,pa6933

def make_grid(pa):
 x0,y0,x1,y1=pa.bounds; rows=[]
 for ix in range(int(np.floor(x0/50000)),int(np.floor(x1/50000))+1):
  for iy in range(int(np.floor(y0/50000)),int(np.floor(y1/50000))+1):
   geom=box(ix*50000,iy*50000,(ix+1)*50000,(iy+1)*50000)
   area=geom.intersection(pa).area
   if area>0: rows.append(('BR-PA',f'{ix}_{iy}',ix,iy,area/1e6,geom))
 geo=gpd.GeoDataFrame(rows,columns=['region_id','block_id','block_ix','block_iy','para_area_km2','geometry'],crs='EPSG:6933').sort_values(['block_ix','block_iy']).reset_index(drop=True)
 canonical=geo[['region_id','block_id','block_ix','block_iy']].copy()
 require(len(canonical)==568,'Unexpected frozen grid block count')
 require(canonical.block_id.is_unique,'Duplicate block IDs')
 require(abs(geo.para_area_km2.sum()-pa.area/1e6)<.01,'Grid does not cover Pará')
 return canonical,geo

def process_source(path,satellite,polygon,valid_blocks):
 transformer=Transformer.from_crs('EPSG:4326','EPSG:6933',always_xy=True)
 selected=[]; report={'source_rows':0,'selected_sensor_rows':0,'para_rows':0,'excluded_outside_para':0,'source_versions':[],'source_date_min':None,'source_date_max':None,'months_present':[],'periods_present':[]}
 versions=set(); months=set(); periods=set(); dates=[]
 for chunk in pd.read_csv(path,chunksize=180000,low_memory=False):
  report['source_rows']+=len(chunk)
  required=['latitude','longitude','acq_date','satellite','daynight','frp','version']
  require(set(required).issubset(chunk.columns),f'{path.name}: missing columns {set(required)-set(chunk.columns)}')
  date=pd.to_datetime(chunk.acq_date,errors='coerce')
  require(date.notna().all(),f'{path.name}: invalid acquisition dates')
  require(date.dt.year.eq(YEAR).all(),f'{path.name}: contains dates outside {YEAR}')
  sat=chunk.satellite.astype(str).str.strip()
  keep=sat.eq(satellite)
  if satellite=='Aqua':require(set(sat.unique())<= {'Aqua','Terra'},'Unexpected MODIS satellite values')
  else:require(keep.all(),'VIIRS source contains non S-NPP records')
  sub=chunk.loc[keep].copy()
  report['selected_sensor_rows']+=len(sub)
  require(sub.daynight.isin(['D','N']).all(),f'{path.name}: invalid daynight')
  require(sub.latitude.between(-90,90).all() and sub.longitude.between(-180,180).all(),'Invalid coordinates')
  mask=intersects_xy(polygon,sub.longitude.to_numpy(dtype=float),sub.latitude.to_numpy(dtype=float))
  report['excluded_outside_para']+=int((~mask).sum())
  sub=sub.loc[mask].copy()
  report['para_rows']+=len(sub)
  if len(sub)==0:continue
  x,y=transformer.transform(sub.longitude.to_numpy(dtype=float),sub.latitude.to_numpy(dtype=float))
  ix=np.floor(x/50000).astype('int64');iy=np.floor(y/50000).astype('int64')
  sub['block_ix']=ix;sub['block_iy']=iy
  sub['block_id']=pd.Series(ix.astype(str),index=sub.index)+'_'+pd.Series(iy.astype(str),index=sub.index)
  require(set(sub.block_id).issubset(valid_blocks),'Clipped points assigned outside canonical grid')
  sub['year']=YEAR;sub['month']=pd.to_datetime(sub.acq_date).dt.month.astype('int64');sub['region_id']='BR-PA'
  sub['frp']=pd.to_numeric(sub.frp,errors='coerce')
  versions.update(sub.version.dropna().astype(str).unique());months.update(sub.month.unique());periods.update(zip(sub.month,sub.daynight))
  dates.extend([sub.acq_date.min(),sub.acq_date.max()])
  selected.append(sub[['region_id','block_id','block_ix','block_iy','year','month','daynight','frp','version']])
 require(selected,f'No Pará detections in {path.name}')
 d=pd.concat(selected,ignore_index=True)
 report.update(source_versions=sorted(versions),source_date_min=min(dates),source_date_max=max(dates),months_present=sorted(map(int,months)),periods_present=[{'month':int(m),'daynight':dn} for m,dn in sorted(periods)])
 require(report['selected_sensor_rows']==report['para_rows']+report['excluded_outside_para'],'Source filtering reconciliation failed')
 return d,report

def aggregate(d,prefix):
 cols=SKEY
 a=d.groupby(cols,as_index=False).agg(count=('frp','size'),frp_sum=('frp',lambda s:s.sum(min_count=1)),version=('version',lambda s:'|'.join(sorted(set(s.dropna().astype(str))))))
 return a.rename(columns={'count':prefix+'_count','frp_sum':prefix+'_frp_sum','version':prefix+'_source_version'})

def load_completeness():
 rows=[{'year':YEAR,'month':m,'daynight':dn,'sensor':sensor,'processing_complete':True,'archive_complete_assumed':False,'archive_evidence':'NASA annual CSV supplied; independent archive completeness not verified','zero_eligible':False} for sensor in ['AQUA_MODIS','SNPP_VIIRS'] for m in range(1,13) for dn in ['D','N']]
 c=pd.DataFrame(rows)
 if COMPLETENESS.exists():
  external=pd.read_csv(COMPLETENESS)
  needed=['year','month','daynight','sensor','processing_complete','archive_complete_assumed','archive_evidence']
  require(set(needed).issubset(external.columns),f'Completeness CSV missing: {set(needed)-set(external.columns)}')
  require(not external.duplicated(['year','month','daynight','sensor']).any(),'Duplicate completeness evidence')
  require(external['archive_evidence'].notna().all(),'Archive evidence must be documented')
  for col in ['processing_complete','archive_complete_assumed']:
   raw=external[col].astype(str).str.lower().str.strip()
   require(raw.isin(['true','false','1','0']).all(),f'Invalid {col} value')
   external[col]=raw.isin(['true','1'])
  require(set(external.sensor).issubset({'AQUA_MODIS','SNPP_VIIRS'}),'Unexpected sensor in completeness CSV')
  require(external.year.eq(YEAR).all() and external.month.between(1,12).all() and external.daynight.isin(['D','N']).all(),'Invalid completeness period')
  external=external[needed]
  c=c.drop(columns=['processing_complete','archive_complete_assumed','archive_evidence','zero_eligible']).merge(external,on=['year','month','daynight','sensor'],how='left',validate='one_to_one')
  c['processing_complete']=c.processing_complete.fillna(False).astype(bool)
  c['archive_complete_assumed']=c.archive_complete_assumed.fillna(False).astype(bool)
  c['archive_evidence']=c.archive_evidence.fillna('No documented archive-completeness evidence')
 c['zero_eligible']=c.processing_complete & c.archive_complete_assumed
 return c

def main():
 for p in [MODIS,VIIRS,BOUNDARY]:
  if not p.exists():raise FileNotFoundError(str(p))
 if OUT.exists():raise FileExistsError(f'Output already exists: {OUT}. Choose a new FC_OUTPUT_DIR; never overwrite.')
 pa4326,pa6933=load_boundary(); canonical,geo=make_grid(pa6933)
 m,m_info=process_source(MODIS,'Aqua',pa4326,set(canonical.block_id))
 v,v_info=process_source(VIIRS,'N',pa4326,set(canonical.block_id))
 require(len(m)>0 and len(v)>0,'No Pará detections for one or both sensors; inspect source availability and satellite selection before export')
 ma=aggregate(m,'modis');va=aggregate(v,'viirs')
 master=ma.merge(va,on=SKEY,how='outer',validate='one_to_one')
 master=master.merge(canonical,on=['region_id','block_id'],how='left',validate='many_to_one',suffixes=('','_grid'))
 require(master[['block_ix','block_iy']].notna().all().all(),'Master contains unmatched canonical grid keys')
 master['coverage_status']='unknown';master['processing_status']='preliminary'
 for c in ['modis_count','viirs_count']:master[c]=master[c].astype('Int64')
 master=master[MASTER_COLS].sort_values(KEY).reset_index(drop=True)
 require(len(master)>0,'Sparse master unexpectedly empty')
 sparse=master[ML_COLS].copy()
 mm=m.groupby(['region_id','block_id','block_ix','block_iy','year','month'],as_index=False).agg(modis_aqua_count=('frp','size'),modis_frp_sum=('frp',lambda s:s.sum(min_count=1)))
 vv=v.groupby(['region_id','block_id','block_ix','block_iy','year','month'],as_index=False).agg(viirs_snpp_count=('frp','size'),viirs_frp_sum=('frp',lambda s:s.sum(min_count=1)))
 bkey=['region_id','block_id','block_ix','block_iy','year','month']
 backend=mm.merge(vv,on=bkey,how='outer',validate='one_to_one')
 a=backend['modis_aqua_count'];b=backend['viirs_snpp_count']
 backend['ratio_viirs_to_modis']=(b/a).where(a.gt(0)&b.gt(0))
 backend['log_ratio']=np.log(backend['ratio_viirs_to_modis'])
 backend=backend[BACK_COLS].sort_values(bkey).reset_index(drop=True)
 coverage=[]
 for sensor,cnt,version,product in [('AQUA_MODIS','modis_count','modis_source_version','MODIS Aqua'),('SNPP_VIIRS','viirs_count','viirs_source_version','VIIRS S-NPP')]:
  d=master[SKEY+[cnt,version]].copy()
  d['sensor']=sensor;d['processing_status']='preliminary';d['coverage_status']='unknown';d['source_product']=product;d['source_version']=d[version]
  d['observation_evidence']=np.where(d[cnt].notna(),'Positive FIRMS detection records present; observation coverage unverified','No positive FIRMS detection for this sensor in sparse union; observation coverage unverified')
  d['notes']='Positive-detection-only archive; no verified clear-sky or fire-free inference'
  coverage.append(d[COV_COLS])
 coverage=pd.concat(coverage,ignore_index=True)
 periods=pd.DataFrame([(YEAR,mo,dn) for mo in range(1,13) for dn in ['D','N']],columns=['year','month','daynight'])
 full=canonical.merge(periods,how='cross').merge(sparse,on=KEY,how='left',validate='one_to_one',indicator=True)
 require(int(full['_merge'].eq('both').sum())==len(sparse),'Sparse records not all present in full panel')
 full=full.drop(columns=['_merge','coverage_status']);full['coverage_status']='unknown'
 comp=load_completeness(); conversions={};nulls={}
 for sensor,cnt in [('AQUA_MODIS','modis_count'),('SNPP_VIIRS','viirs_count')]:
  eligible=comp.loc[comp.sensor.eq(sensor),['year','month','daynight','zero_eligible']].rename(columns={'zero_eligible':'eligible'})
  full=full.merge(eligible,on=['year','month','daynight'],how='left',validate='many_to_one')
  allowed=full.eligible.astype('boolean').fillna(False).astype(bool)
  before=full[cnt].copy().astype('Int64')
  mask=allowed & before.isna()
  conversions[sensor]=int(mask.sum());full.loc[mask,cnt]=0
  full[cnt]=full[cnt].astype('Int64');full=full.drop(columns='eligible')
  nulls[sensor]=int(full[cnt].isna().sum())
  require(int(full[cnt].sum())==int(master[cnt].sum()),f'Changed {sensor} detection totals')
  original=full.merge(sparse[KEY+[cnt]],on=KEY,how='inner',validate='one_to_one',suffixes=('_full','_original'))
  orig=original[cnt+'_original'];now=original[cnt+'_full'];pos=orig.notna()
  bad=original.loc[pos & now.ne(orig).fillna(True)]
  require(bad.empty,f'Changed original positive {sensor} counts: {bad.head(3).to_dict("records")}')
  require(int((now.eq(0).fillna(False)&pos).sum())==0,'Original positive counts were zeroed')
 full=full[ML_COLS].sort_values(KEY).reset_index(drop=True)
 require(len(full)==len(canonical)*12*2,'Full panel must contain every canonical block x month x D/N combination')
 require(full.groupby('block_id').size().eq(24).all(),'Each block must have 24 periods')
 require(not full.duplicated(KEY).any() and not master.duplicated(KEY).any() and not backend.duplicated(bkey).any(),'Duplicate keys')
 require(master.columns.tolist()==MASTER_COLS and sparse.columns.tolist()==ML_COLS and backend.columns.tolist()==BACK_COLS and full.columns.tolist()==ML_COLS and coverage.columns.tolist()==COV_COLS,'Schema mismatch')
 require(full.coverage_status.eq('unknown').all() and master.coverage_status.eq('unknown').all() and coverage.coverage_status.eq('unknown').all(),'Coverage label changed')
 require(int(master.modis_count.sum())==len(m) and int(master.viirs_count.sum())==len(v),'Source-to-master detection totals mismatch')
 # Write all artifacts in a staging folder; publish only when all validations and writes succeed.
 parent=OUT.parent;parent.mkdir(parents=True,exist_ok=True)
 stage=Path(tempfile.mkdtemp(prefix=f'firechronicle_{YEAR}_staging_',dir=parent))
 try:
  fmt=os.environ.get('FC_EXPORT_FORMAT','parquet')
  if fmt=='parquet':
   import pyarrow # required for real Parquet output
  elif fmt!='csv':raise ValueError('FC_EXPORT_FORMAT must be parquet or csv')
  frames={
   f'firechronicle_master_{YEAR}.parquet':master,
   f'overlap_training_{YEAR}.parquet':sparse,
   f'backend_monthly_{YEAR}.parquet':backend,
   f'coverage_para_{YEAR}.parquet':coverage,
   f'overlap_training_full_panel_{YEAR}.parquet':full,
   'para_50km_grid_canonical.parquet':canonical,
   'para_50km_grid_geometry.parquet':geo,
  }
  for name,df in frames.items():
   if fmt=='parquet':df.to_parquet(stage/name,index=False)
   else:
    if name.endswith('grid_geometry.parquet'):
     d=df.drop(columns='geometry').copy();d['geometry_wkt']=df.geometry.to_wkt();d.to_csv(stage/name.replace('.parquet','.csv'),index=False)
    else:df.to_csv(stage/name.replace('.parquet','.csv'),index=False)
  comp.to_csv(stage/f'source_period_completeness_{YEAR}.csv',index=False)
  inputs={p.name:{'bytes':p.stat().st_size,'sha256':sha(p)} for p in [MODIS,VIIRS,BOUNDARY]}
  audit={
   'export_format':fmt,'year':YEAR,'created_utc':datetime.now(timezone.utc).isoformat(),'region_id':'BR-PA',
   'source':{'modis':m_info,'viirs_snpp':v_info},'input_files':inputs,
   'boundary':{'source':'IBGE PA_UF_2024.zip','source_crs':'EPSG:4674','grid_crs':'EPSG:6933','block_size_m':50000,'origin_m':[0,0],'edge_policy':'point intersects Pará polygon','block_policy':'positive-area intersection with Pará','area_km2':float(geo.para_area_km2.sum())},
   'rows':{'canonical_grid':len(canonical),'master':len(master),'sparse_ml':len(sparse),'backend_monthly':len(backend),'coverage':len(coverage),'full_panel_ml':len(full)},
   'counts':{'aqua_modis':int(master.modis_count.sum()),'snpp_viirs':int(master.viirs_count.sum())},
   'completeness':{'evidence_rows':len(comp),'archive_completeness_independently_verified':False,'eligible_sensor_periods':int(comp.zero_eligible.sum()),'recorded_zero_conversions':conversions,'remaining_null_counts':nulls,'interpretation':'Zeros are recorded-detection zeros under documented archive-completeness assumptions, not verified fire-free or clear-sky observations. Without archive evidence, NULLs remain.'},
   'coverage_status':'unknown','processing_status':'preliminary',
   'schemas':{'master':MASTER_COLS,'sparse_ml':ML_COLS,'backend':BACK_COLS,'full_panel_ml':ML_COLS,'coverage':COV_COLS},
   'validation':{'passed':True,'positive_counts_preserved':True,'no_duplicate_keys':True,'all_blocks_have_24_periods':True,'full_panel_complete':len(full)==len(canonical)*24,'grid_568':len(canonical)==568},
  }
  audit['output_files']={p.name:{'bytes':p.stat().st_size,'sha256':sha(p)} for p in sorted(stage.iterdir()) if p.is_file()}
  (stage/f'audit_{YEAR}.json').write_text(json.dumps(audit,indent=2,default=str))
  (stage/'README.txt').write_text(f'FireChronicle {YEAR} Pará exports. Source: Brazil NASA FIRMS Aqua MODIS and S-NPP VIIRS CSVs; IBGE Pará 2024 boundary. Coverage unverified; zeros only if completeness evidence explicitly authorizes them. See audit_{YEAR}.json.\n')
  zipname=f'FireChronicle_{YEAR}_Validated_Outputs.zip' if fmt=='parquet' else f'FireChronicle_{YEAR}_Validated_CSV_Preview.zip'
  with zipfile.ZipFile(stage/zipname,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
   for p in sorted(stage.iterdir()):
    if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
  stage.rename(OUT)
  print('SUCCESS: verified outputs:',OUT)
  print(json.dumps({'rows':audit['rows'],'counts':audit['counts'],'completeness':audit['completeness']},indent=2))
  return audit
 except Exception:
  shutil.rmtree(stage,ignore_errors=True)
  raise


## 5. Run the selected year

This calls the pipeline's `main()` function using the year and paths configured above.

The pipeline checks that the output folder does not already exist. This protects previously generated results from accidental overwriting.

In [ ]:
audit = main()
print("\nAnnual processing finished.")
print("Validation passed:", audit["validation"]["passed"])
print("Audit file:", Path(OUTPUT_ROOT) / f"audit_{YEAR}.json")

## 6. Download the annual package and audit report

The ZIP contains the exported datasets and a README. The JSON audit records the input and output checksums, row counts, completeness handling, and validation results.

Keep the audit report with the package so the annual output can be traced to its source files.

In [ ]:
from pathlib import Path
from google.colab import files

output_dir = Path(OUTPUT_ROOT)
zip_name = (
    f"FireChronicle_{YEAR}_Validated_Outputs.zip"
    if EXPORT_FORMAT == "parquet"
    else f"FireChronicle_{YEAR}_Validated_CSV_Preview.zip"
)
zip_path = output_dir / zip_name
audit_path = output_dir / f"audit_{YEAR}.json"

assert zip_path.is_file(), f"Annual package not found: {zip_path}"
assert audit_path.is_file(), f"Audit report not found: {audit_path}"

print("Annual package:", zip_path.name)
print("Audit report:", audit_path.name)

files.download(str(zip_path))
files.download(str(audit_path))

## Output files

For Parquet export, the annual package includes:

| File | Purpose |
|---|---|
| `firechronicle_master_YEAR.parquet` | Sensor-specific monthly master records |
| `overlap_training_YEAR.parquet` | Sparse MODIS–VIIRS training/overlap records |
| `backend_monthly_YEAR.parquet` | Monthly backend aggregates and ratios |
| `coverage_para_YEAR.parquet` | Sensor and period coverage metadata |
| `overlap_training_full_panel_YEAR.parquet` | Full block × month × day/night panel |
| `para_50km_grid_canonical.parquet` | Canonical block identifiers |
| `para_50km_grid_geometry.parquet` | Grid geometry and block areas |
| `source_period_completeness_YEAR.csv` | Completeness assumptions and evidence |
| `audit_YEAR.json` | Validation and provenance report |

## Reproducibility note

Notebook outputs are cleared in the GitHub copy so that old Colab runtime messages are not mistaken for a fresh run. Execute the notebook in Colab to generate current outputs and a new audit report.

The annual pipeline validates the supplied data and its internal consistency. It does not independently establish that every sensor archive period is complete or that an area was fire-free when no detections were recorded.